In [ ]:
import torch
import torch.nn as nn
import os
import sys
import time
import numpy as np
import pandas as pd
import gc
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

sys.path.append("..")
# --- Project Imports ---
from src.utility.config import BASELINE_MODEL_PATH, QUANTIZED_MODELS, NUM_CLASSES
from src.utility.utils import get_data_loaders
from src.utility.quantization_calibration import calibrate_model
from src.model import CNN
from src.layers import QuantizedLayerMixin

# ==========================================
# 1. UTILITY FUNCTIONS
# ==========================================

DEVICE = 'cpu'

def load_baseline_model(model_path):
    """
    Loads FP32 model. Handles class mismatches (150 vs 3) gracefully
    by skipping layers that don't match the shape.
    """
    print(f"Loading Baseline: {model_path}")
    
    # 1. Initialize the model shell (e.g. 3 classes)
    model = CNN(num_classes=NUM_CLASSES).to('cpu') 
    
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"{model_path} not found.")
    
    # 2. Load the file
    state_dict = torch.load(model_path, map_location='cpu')
    
    # 3. SMART FILTER: Remove keys that have different shapes (e.g. fc2 with 150 vs 3)
    model_state = model.state_dict()
    filtered_state_dict = {}
    
    for k, v in state_dict.items():
        if k in model_state:
            if v.shape == model_state[k].shape:
                filtered_state_dict[k] = v
            else:
                print(f"  -> WARNING: Skipping layer '{k}' due to shape mismatch "
                      f"(File: {v.shape} vs Model: {model_state[k].shape})")
    
    # 4. Load with strict=False to ignore missing quant buffers and skipped layers
    model.load_state_dict(filtered_state_dict, strict=False)
    model.eval()
    return model

def load_quantized_inference_model(path, num_classes):
    """
    Loads the saved Int8 state and reconstructs the CPU-optimized kernels.
    """
    # Initialize architecture
    model = CNN(num_classes=num_classes)
    state_dict = torch.load(path, map_location='cpu')
    
    # repare the model structure to match the state_dict shapes
    print("Preparing model buffers...")
    for name, module in model.named_modules():
        if hasattr(module, "prepare_integer_inference"):
            # Register weight_int8 with the correct shape from the file
            w_int8_key = f"{name}.weight_int8"
            if w_int8_key in state_dict:
                shape = state_dict[w_int8_key].shape
                module.register_buffer('weight_int8', torch.zeros(shape, dtype=torch.int8))
            
            # Match the empty weight shape [0] used during saving
            module.weight = torch.nn.Parameter(torch.empty(0), requires_grad=False)
            
            # Handle Bias (Required for prepacking)
            bias_key = f"{name}.bias"
            if bias_key in state_dict:
                b_shape = state_dict[bias_key].shape
                module.bias = torch.nn.Parameter(torch.zeros(b_shape), requires_grad=False)

    # 3. Load the data (scales, ZPs, and int8 weights)
    model.load_state_dict(state_dict, strict=False)
    
    # 4. RECONSTRUCTION LOOP
    print("Reconstructing CPU optimized kernels...")
    for name, module in model.named_modules():
        # Check if it is one of our custom quantized layers
        if hasattr(module, "prepare_integer_inference"):

            # Temporarily restore float weight for the prepacker
            w_float = module.weight_int8.float() * module.weight_scale
            module.weight = torch.nn.Parameter(w_float, requires_grad=False)
            
            # Manually settings flags for packing
            module.quant_mode = True
            module.activation_calibrated = True # The prepacker needs this check to pass idk why
            
            # This populates module.packed_params with the C++ capsule
            module.prepare_integer_inference()
            
            # Verify packing succeeded
            if module.packed_params is None:
                print(f"Warning: Packing failed for {name}")
            
            # Set final inference flag
            module.int_mode = True
            
            # Remove float weight to save RAM
            module.weight = torch.nn.Parameter(torch.empty(0), requires_grad=False)
            
    model.eval()
    return model

def measure_performance(model, loader, device='cpu', num_batches=100):
    """
    Measures Latency (ms) and Throughput (FPS/img per sec).
    """
    model.eval()
    model.to(device)
    
    # 1. Warmup (gets CPU cache ready)
    print("   -> Warming up...", end="\r")
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= 5: break
            _ = model(imgs.to(device))
            
    # 2. Measurement
    print(f"   -> Benchmarking ({num_batches} batches)...", end="\r")
    start_time = time.time()
    total_samples = 0
    
    with torch.no_grad():
        for i, (imgs, _) in enumerate(loader):
            if i >= num_batches: break
            imgs = imgs.to(device)
            _ = model(imgs)
            total_samples += imgs.size(0)
            
    end_time = time.time()
    total_time = end_time - start_time
    
    latency_ms = (total_time / total_samples) * 1000
    fps = total_samples / total_time
    print(f"   -> Done. {total_samples} samples processed.")
    
    return latency_ms, fps

def evaluate_metrics(model, loader, device='cpu'):
    """
    Returns a dictionary with Accuracy, Precision, Recall, and F1-Score.
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    # Calculate metrics (weighted average handles class imbalance)
    acc = accuracy_score(all_targets, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average='weighted', zero_division=0
    )
    
    return {
        "Accuracy": acc * 100,
        "F1-Score": f1 * 100,
        "Precision": precision * 100,
        "Recall": recall * 100
    }

def get_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

def check_all_layers_quantization(model):
    print("\n=== DETAILED LAYER VERIFICATION ===")
    print(f"{'Layer Name':<25} | {'Status':<10} | {'Dtype':<15} | {'Range (Min/Max)'}")
    print("-" * 80)

    for name, module in model.named_modules():
        # 1. Filter for relevant layers (Conv, Linear) or layers with Int8 weights
        if isinstance(module, (nn.Conv2d, nn.Linear)) or hasattr(module, 'weight_int8'):
            
            status = "MISSING"
            dtype_str = "N/A"
            range_str = "N/A"

            # 2. Check if the specific Int8 buffer exists and is not None
            if hasattr(module, 'weight_int8') and module.weight_int8 is not None:
                w = module.weight_int8
                
                # 3. Verify Dtype
                if w.dtype == torch.int8:
                    status = "INT8 (OK)"
                    dtype_str = str(w.dtype)
                    range_str = f"{w.min().item()} / {w.max().item()}"
                else:
                    status = "WRONG TYPE"
                    dtype_str = str(w.dtype)
            
            # 4. Fallback: Check if it is still Float32 (not quantized)
            elif hasattr(module, 'weight') and module.weight is not None:
                if module.weight.numel() > 0:
                    status = "FP32 (Ref)"
                    dtype_str = "float32"
                else:
                    status = "EMPTY (0)" # Weights stripped but Int8 missing

            print(f"{name:<25} | {status:<10} | {dtype_str:<15} | {range_str}")

    print("-" * 80)

In [34]:
gc.collect()
torch.cuda.empty_cache()

print("=== 1. SETUP & BASELINE ===")
# Force CPU for fair Int8 comparison
torch.backends.quantized.engine = 'fbgemm' 
train_loader, test_loader, num_classes = get_data_loaders()

# --- A. Baseline Evaluation ---
print("\n[Baseline] Loading FP32 Model...")
baseline_model = load_baseline_model(BASELINE_MODEL_PATH)
base_size = get_size_mb(BASELINE_MODEL_PATH)

print("[Baseline] Measuring Performance...")
base_lat, base_fps = measure_performance(baseline_model, test_loader)
print("[Baseline] Calculating Metrics...")
base_metrics = evaluate_metrics(baseline_model, test_loader)

=== 1. SETUP & BASELINE ===

[Baseline] Loading FP32 Model...
Loading Baseline: /home/dominic/Desktop/Research-Project/Forschungsprojekt/results/models/baseline_float32.pt
[Baseline] Measuring Performance...
   -> Done. 1364 samples processed.
[Baseline] Calculating Metrics...


In [35]:
print("\n=== 2. QUANTIZATION (SYMMETRIC INT8) ===")

# 1. Quantize Weights
print("[Quantization] Quantizing Weights...")
for name, module in baseline_model.named_modules():
    if isinstance(module, QuantizedLayerMixin):
        module.prepare_quantization(method='symmetric', bits=8)

# 2. Calibrate (Determine Scale/ZeroPoint)
print("[Quantization] Calibrating (10 batches)...")
calibrate_model(baseline_model, test_loader, num_batches=10, device=DEVICE)

# 3. Convert (Switch to Int8 inference mode)
print("[Quantization] Converting to Integer Mode...")
for name, module in baseline_model.named_modules():
    if isinstance(module, QuantizedLayerMixin):
        # We keep the 'act_scale' from calibration, 
        # but we shift the 'zero_point' to 128 for CPU compatibility.
        module.act_zero_point.fill_(128)
        module.out_zero_point.fill_(128)
        
        # The weight_zero_point MUST remain 0 to stay Symmetric.
        module.weight_zero_point.fill_(0)

# 4. Transition to Integer Mode (Now with ZP=128 for CPU symmetry)
print("Preparing CPU Kernels...")
for name, module in baseline_model.named_modules():
    if hasattr(module, "prepare_integer_inference"):
        module.prepare_integer_inference()

# 5. Save to Disk
save_path = os.path.join(QUANTIZED_MODELS, "custom_symmetric_cpu_int8.pt")
torch.save(baseline_model.state_dict(), save_path)
quant_size = get_size_mb(save_path)
print(f"[Quantization] Model saved to {save_path}")




=== 2. QUANTIZATION (SYMMETRIC INT8) ===
[Quantization] Quantizing Weights...
[Quantization] Calibrating (10 batches)...
[Quantization] Converting to Integer Mode...
Preparing CPU Kernels...
[Quantization] Model saved to /home/dominic/Desktop/Research-Project/Forschungsprojekt/results/quantized_models/custom_symmetric_cpu_int8.pt


In [50]:
print("\n=== 3. VERIFICATION (RELOAD FROM DISK) ===")

# --- C. Reload & Re-Activate ---
# This proves the file on disk works
conf = {'method': 'symmetric', 'bits': 8}

# Übergib diese Config an load_model
quant_model = load_quantized_inference_model(save_path, NUM_CLASSES)

# print("[Verification] Re-activating Integer Buffers...")
# for name, module in quant_model.named_modules():
#     if hasattr(module, "prepare_integer_inference"):
#         # CRITICAL: This restores the link to 'weight_int8' after loading
#         module.prepare_integer_inference()
#         if hasattr(module, 'weight_zero_point'):
#             module.weight_zero_point.fill_(0)

print("[Verification] Measuring Performance...")
quant_lat, quant_fps = measure_performance(quant_model, test_loader)
print("[Verification] Calculating Metrics...")
quant_metrics = evaluate_metrics(quant_model, test_loader)


=== 3. VERIFICATION (RELOAD FROM DISK) ===
Preparing model buffers...
Reconstructing CPU optimized kernels...
[Verification] Measuring Performance...
   -> Done. 1364 samples processed.
[Verification] Calculating Metrics...


In [ ]:
# 4. FINAL REPORT

print("\n" + "="*60)
print("                 FINAL RESEARCH RESULTS")
print("="*60)

results = {
    "Metric": [
        "Accuracy (%)", "F1-Score (%)", "Precision (%)", "Recall (%)", 
        "Throughput (img/sec)", "Latency (ms)", "Size (MB)"
    ],
    "Baseline (FP32)": [
        base_metrics["Accuracy"], base_metrics["F1-Score"], 
        base_metrics["Precision"], base_metrics["Recall"],
        base_fps, base_lat, base_size
    ],
    "Quantized (Int8)": [
        quant_metrics["Accuracy"], quant_metrics["F1-Score"], 
        quant_metrics["Precision"], quant_metrics["Recall"],
        quant_fps, quant_lat, quant_size
    ]
}

df = pd.DataFrame(results)

# Add Improvement Column
df["Delta / Speedup"] = [
    f"{df.iloc[0, 2] - df.iloc[0, 1]:.5f}", # Accuracy Delta
    f"{df.iloc[1, 2] - df.iloc[1, 1]:.5f}", # F1 Delta
    f"{df.iloc[2, 2] - df.iloc[2, 1]:.5f}", # Precision Delta
    f"{df.iloc[3, 2] - df.iloc[3, 1]:.5f}", # Recall Delta
    f"{df.iloc[4, 2] / df.iloc[4, 1]:.5f}x",# Throughput Speedup
    f"{df.iloc[5, 1] / df.iloc[5, 2]:.5f}x",# Latency Speedup
    f"{df.iloc[6, 1] / df.iloc[6, 2]:.5f}x" # Size Compression
]

print(df.to_string(index=False))
print("="*60)

# Quick Layer Check
# 5. Layer Verify
print("\n=== DETAILED LAYER VERIFICATION ===")
print(f"{'Layer Name':<20} | {'Status':<15} | {'Dtype':<15} | {'Range (Min/Max)'}")
print("-" * 80)

for name, module in quant_model.named_modules():
    if hasattr(module, 'weight_int8'):
        status = "MISSING"
        dtype_str = "N/A"
        range_str = "N/A"
        
        # Check buffer content
        if module.weight_int8 is not None:
             # Check if it's actually populated (not all zeros if that's impossible, but mainly checking dtype)
            w = module.weight_int8
            dtype_str = str(w.dtype)
            
            if w.dtype == torch.int8:
                status = "INT8"
                range_str = f"{w.min()}/{w.max()}"
            else:
                status = "WRONG TYPE"
        
        print(f"{name:<20} | {status:<15} | {dtype_str:<15} | {range_str}")


                 FINAL RESEARCH RESULTS
              Metric  Baseline (FP32)  Quantized (Int8) Delta / Speedup
        Accuracy (%)        90.909091         90.909091         0.00000
        F1-Score (%)        90.898742         90.912625         0.01388
       Precision (%)        92.074878         92.049988        -0.02489
          Recall (%)        90.909091         90.909091         0.00000
Throughput (img/sec)       175.018137        275.081212        1.57173x
        Latency (ms)         5.713694          3.635290        1.57173x
           Size (MB)         6.083360          1.538995        3.95281x

=== DETAILED LAYER VERIFICATION ===
Layer Name           | Status          | Dtype           | Range (Min/Max)
--------------------------------------------------------------------------------
conv1                | INT8 (OK)       | torch.int8      | -127/97
conv2                | INT8 (OK)       | torch.int8      | -119/127
conv3                | INT8 (OK)       | torch.int8    